# Day 02 · 고급 검색 전략 실습 (로컬)
Day 01 RAG 봇에 **하이브리드(BM25+벡터)·Reranker·HyDE·Multi-Query·미니 GraphRAG**를 얹어 검색 품질을 끌어올립니다.
검색(BM25·벡터·reranker)은 전부 CPU 로컬 — LLM(Qwen)은 **생성·질의 변환·관계 추출**에만 사용합니다.

## 0 · 설치 & 설정

In [ ]:
!pip install -q sentence-transformers qdrant-client openai rank_bm25 networkx

In [ ]:
import os, getpass
from openai import OpenAI

# LLM 엔드포인트: NVIDIA build (기본). 강사 DGX로 바꾸려면 아래 줄 주석 해제.
QWEN_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
# QWEN_BASE_URL = "http://124.51.229.210:30001/v1"   # ← 강사 DGX 백업
QWEN_API_KEY  = os.getenv("NVIDIA_API_KEY") or os.getenv("QWEN_API_KEY") or getpass.getpass("NVIDIA API 키(nvapi-...) 입력: ")

_c = OpenAI(base_url=QWEN_BASE_URL, api_key=QWEN_API_KEY)
_av = [m.id for m in _c.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl", "embed", "rerank"))]
    LLM_MODEL = _q[0] if _q else _av[0]
print("엔드포인트:", QWEN_BASE_URL, "| 모델:", LLM_MODEL)

EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

## 1 · Day 01 검색 준비 (요약 재구성)
청크·임베딩·로컬 Qdrant까지 한 번에 세팅합니다.

> 문서가 **6개뿐이라 N=6 → K=3**으로 축소해 실습합니다. 실전 권장은 슬라이드처럼 **N≈30~50 → K≈3~6**.

In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

texts = [
    "연차 휴가는 사용 3영업일 전까지 사내포털에서 신청하고 팀장 승인을 받는다.",
    "비용 정산은 영수증 첨부 후 경영지원팀 검토, 재무팀이 최종 승인한다.",
    "재택근무는 주 2회까지 가능하며 전날 18시까지 팀 채널에 공유한다.",
    "에러코드 ERR-4021은 인증 토큰 만료를 의미하며 재로그인으로 해결한다.",
    "제품 X-200의 펌웨어 최신 버전은 2.3이며 설정 메뉴에서 업데이트한다.",
    "보안상 외부 USB는 금지이며 파일 공유는 사내 드라이브를 사용한다.",
]

embedder = SentenceTransformer(EMBED_MODEL)   # 최초 1회 모델 다운로드로 시간이 걸릴 수 있어요
def embed(xs): return embedder.encode(xs, normalize_embeddings=True)

client = QdrantClient(":memory:")
dim = embed(["x"]).shape[1]
if client.collection_exists("docs"):
    client.delete_collection("docs")
client.create_collection("docs", vectors_config=VectorParams(size=dim, distance=Distance.COSINE))

vectors = embed(texts)                         # 한 번에 배치 임베딩
client.upsert("docs", points=[PointStruct(id=i, vector=vectors[i].tolist(), payload={"text": t})
                              for i, t in enumerate(texts)])

def vector_search_idx(query, n=10):
    qv = embed([query])[0]
    return [p.id for p in client.query_points("docs", query=qv.tolist(), limit=n).points]

def show(idxs):                                # 인덱스 → 내용 미리보기 (결과 확인용)
    return [texts[i][:18] for i in idxs]

print("준비 완료:", len(texts), "청크")

## Lab 1 · BM25 (키워드 검색)
정확 토큰(ERR-4021, X-200)에 강한 색인 검색.

✅ **확인**: `ERR-4021`이 BM25 1위로 잡히는가?

In [ ]:
import re
from rank_bm25 import BM25Okapi

# 한국어는 조사가 붙어 단순 split()이면 'ERR-4021은'·'X-200의' 같은 토큰이 되어
# 쿼리 'ERR-4021'·'X-200'과 매칭되지 않습니다(점수 0).
# 영문/숫자/하이픈 덩어리와 한글 덩어리를 분리해 '정확 토큰'을 살립니다.
# (단, '환불은'처럼 순수 한글 단어의 조사는 그대로 붙습니다 → 형태소 분석기(kiwipiepy)로 개선 가능)
def tokenize(t):
    return re.findall(r"[A-Za-z0-9\-]+|[가-힣]+", t)

tokenized = [tokenize(t) for t in texts]
bm25 = BM25Okapi(tokenized)

def bm25_search(query, k=10):
    scores = bm25.get_scores(tokenize(query))
    return sorted(range(len(scores)), key=lambda i: -scores[i])[:k]

print("BM25 'ERR-4021':", show(bm25_search("ERR-4021", 3)))

## Lab 2 · 하이브리드 + RRF
벡터·BM25 결과를 **순위(rank)** 로 합칩니다. k=60 (슬라이드 손계산과 같은 공식: 1위 → 1/61).

✅ **확인**: 벡터만 / BM25만 / 하이브리드의 1위가 어떻게 다른가?

In [ ]:
def rrf(rank_lists, k=60):
    score = {}
    for ranks in rank_lists:
        for r, idx in enumerate(ranks):
            score[idx] = score.get(idx, 0) + 1/(k + r + 1)
    return sorted(score, key=lambda i: -score[i])

def hybrid(query, n=6):
    return rrf([vector_search_idx(query, n), bm25_search(query, n)])[:n]

for tag, fn in [("벡터", vector_search_idx), ("BM25", bm25_search), ("하이브리드", hybrid)]:
    print(f"{tag:5s} →", show(fn("X-200 펌웨어 버전", 3)))

## Lab 3 · Reranker (서류전형 → 면접)
cross-encoder가 (질문, 문서)를 **함께** 읽어 관련성으로 다시 줄세웁니다. N=후보 → K=최종.

✅ **확인**: rerank 전/후 Top-3 순서가 어떻게 바뀌는가?

In [ ]:
from sentence_transformers import CrossEncoder
# 다국어 reranker (한국어 지원). 영어 전용 ms-marco-MiniLM 은 한국어에서 오작동합니다.
# 최초 1회 모델 다운로드로 몇십 초 걸릴 수 있어요 — 멈춘 게 아닙니다!
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")   # 로컬·다국어

def rerank(query, cand_idx, top_k=3):
    pairs = [(query, texts[i]) for i in cand_idx]
    scores = reranker.predict(pairs)
    order = sorted(range(len(cand_idx)), key=lambda j: -scores[j])
    return [cand_idx[j] for j in order[:top_k]]

q = "비용은 누가 최종 승인하나요?"
cand = hybrid(q, n=6)              # 1차: 넓게 (서류전형)
best = rerank(q, cand, top_k=3)    # 2차: 정밀 (면접)
print("rerank 전:", show(cand))
print("rerank 후:", show(best))

## Lab 4 · HyDE (가상 답변으로 검색)
짧고 모호한 질문에 '그럴듯한 가상 답'을 만들어 **그걸로** 검색합니다.

✅ **확인**: 같은 모호 질문에서 '질문 그대로' vs 'HyDE'의 결과 차이

In [ ]:
from openai import OpenAI
llm = OpenAI(base_url=QWEN_BASE_URL, api_key=QWEN_API_KEY)

def ask(prompt, temperature=0.3):
    out = llm.chat.completions.create(model=LLM_MODEL, temperature=temperature,
        messages=[{"role": "user", "content": prompt}]).choices[0].message.content
    return out.split("</think>")[-1].strip()   # 모델이 <think>를 출력하는 설정이어도 안전

def hyde_search(query, n=5):
    hypo = ask(f"다음 질문에 짧게 그럴듯한 답을 써줘:\n{query}")
    return vector_search_idx(hypo, n)

q = "그거 며칠 전에 말해야 해요?"          # 짧고 모호한 질문
print("질문 그대로:", show(vector_search_idx(q, 3)))
print("HyDE 검색  :", show(hyde_search(q, 3)))

## Lab 5 · Multi-Query (여러 표현으로 검색)
질문을 여러 표현으로 바꿔 각각 검색 → RRF로 합칩니다. **원 질문도 목록에 포함**하는 게 관행입니다.

✅ **확인**: 변형 질의들이 서로 다른 청크를 데려오는가?

In [ ]:
def multi_query(query, n=6, verbose=True):
    raw = ask(f"질문을 뜻은 같되 표현이 다른 3개로 바꿔, 한 줄씩만 적어줘:\n{query}", temperature=0.7)
    # LLM이 붙이는 번호("1. ")·서두 문장("다음은 ~:")을 걷어냅니다
    lines = [re.sub(r'^\s*[\d\-\.\)\*]+\s*', '', l).strip() for l in raw.splitlines()]
    variants = [query] + [l for l in lines if l and not l.endswith(':')][:3]
    if verbose:
        print("검색에 쓸 표현들:", variants)
    return rrf([vector_search_idx(v, n) for v in variants])[:n]

print("결과:", show(multi_query("재택 언제까지 알려줘야 하나요?")))

## Lab 6 · 최종 파이프라인 & A/B 비교 (질문 8개)
**하이브리드(N) → reranker → Top-K → 생성**. 모호 질문은 `vague=True`로 **앞단에 HyDE·Multi-Query**를 더합니다.

슬라이드 기준 질문 8개 = **정확 토큰형 3 · 의미형 3 · 모호형 2**. LLM 호출이 많아 **2~3분** 걸릴 수 있어요.

✅ **확인**: 각 질문에서 기준선 vs 고급 — 어느 쪽 근거가 정확했는지 옆에 메모

In [ ]:
SYSTEM = ("아래 [자료]의 내용만 근거로 답하라. 없으면 '문서에 없음'이라고만 답하라. "
          "답 끝에 (근거)를 표기하라.")

def generate(query, idxs):
    ctx = "\n".join(f"[{i+1}] {texts[d]}" for i, d in enumerate(idxs))
    out = llm.chat.completions.create(model=LLM_MODEL, temperature=0.2,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": f"[자료]\n{ctx}\n\n[질문] {query}"}])
    return out.choices[0].message.content.split("</think>")[-1].strip()

def advanced_rag(query, vague=False):
    cand = hybrid(query, n=6)
    if vague:   # 모호 질문: HyDE·Multi-Query로 후보를 보강 (파이프라인 '맨 앞' 블록)
        cand = rrf([cand, hyde_search(query, 6), multi_query(query, 6, verbose=False)])[:6]
    best = rerank(query, cand, top_k=3)        # 재정렬은 항상 '원 질문' 기준
    return generate(query, best)

def baseline_rag(query):                        # Day 1 벡터-only 기준선
    return generate(query, vector_search_idx(query, 3))

In [ ]:
QUESTIONS = {
    "정확 토큰형": ["에러코드 ERR-4021 해결법은?", "X-200 펌웨어 최신 버전은?", "외부 USB 써도 되나요?"],
    "의미형":     ["로그인이 자꾸 풀리는데 왜죠?", "비용은 누가 최종 승인하나요?", "집에서 일하려면 어떻게 해야 하나요?"],
    "모호형":     ["그거 며칠 전에 말해야 해요?", "파일은 어디로 공유해요?"],
}

for cat, qs in QUESTIONS.items():
    for q in qs:
        print(f"\n◆ [{cat}] {q}")
        print("  기준선:", baseline_rag(q))
        print("  고급  :", advanced_rag(q, vague=(cat == "모호형")))

## Lab 7 · 미니 GraphRAG (관계를 뽑아 연결)
슬라이드의 **'그래프감' 판별**을 통과한 데이터로 지식그래프를 만듭니다 — 고유명사가 많고,
'누가·무엇과 어떻게'를 묻게 되고, 답이 여러 문서에 흩어진 데이터.

흐름: ① LLM 트리플 추출(JSON) → ② networkx 그래프 → ③ 질문 속 엔티티에서 **2-hop 이웃 탐색** → ④ 경로를 근거로 답변

In [ ]:
import json
import networkx as nx

graph_docs = [
    "제품 X-200은 아크메전자가 제조한다.",
    "아크메전자의 대표이사는 김철수다.",
    "X-200의 배터리는 파워셀이 공급한다.",
    "김철수는 2024년 혁신경영 대상을 수상했다.",
]

def extract_triples(docs):
    prompt = ("각 문장에서 (주체, 관계, 대상) 트리플을 뽑아 JSON 배열로만 출력해. 설명 금지.\n"
              '예: [["X-200","제조사","아크메전자"]]\n\n' + "\n".join(docs))
    for attempt in range(2):                        # 파싱 실패 시 1회 재시도 (트러블슈팅 표 참고)
        raw = ask(prompt, temperature=0).strip()
        if raw.startswith("```"):                   # ```json ... ``` 코드펜스 제거
            raw = raw.strip("`").strip()
            if raw[:4].lower() == "json":
                raw = raw[4:].strip()
        try:
            return [tuple(t) for t in json.loads(raw)]
        except json.JSONDecodeError:
            print(f"  JSON 파싱 실패(시도 {attempt+1}) → 재시도")
    raise RuntimeError("트리플 추출 실패 — 프롬프트의 'JSON 배열만' 지시를 확인하세요")

triples = extract_triples(graph_docs)
print("추출된 트리플:", triples)

G = nx.DiGraph()
for s, r, o in triples:
    G.add_edge(s, o, rel=r)
print("노드:", list(G.nodes))

In [ ]:
def neighbors_context(G, start, hops=2):        # start에서 hops걸음 안의 관계(길)를 모두 수집
    seen, frontier, lines = {start}, {start}, []
    for _ in range(hops):
        nxt = set()
        for u in frontier:
            for _, v, d in G.out_edges(u, data=True):
                lines.append(f"{u} -[{d['rel']}]-> {v}"); nxt.add(v)
            for v, _, d in G.in_edges(u, data=True):
                lines.append(f"{v} -[{d['rel']}]-> {u}"); nxt.add(v)
        frontier = nxt - seen; seen |= nxt
    return "\n".join(dict.fromkeys(lines))

def graph_answer(query, hops=2):
    start = max((n for n in G.nodes if n in query), key=len, default=None)   # 질문 속 엔티티 찾기
    if start is None:
        return "질문에서 그래프 엔티티를 찾지 못함"
    ctx = neighbors_context(G, start, hops)
    print(f"[{start}에서 {hops}-hop 경로]\n{ctx}\n")
    return ask(f"[관계 정보]\n{ctx}\n\n[질문] {query}\n관계 정보만 근거로 짧게 답해줘.", temperature=0)

q = "X-200 제조사의 대표는 누구인가요?"     # 멀티홉 질문 — 한 문장엔 답이 없다!
print("그래프 답:", graph_answer(q))

In [ ]:
# 같은 멀티홉 질문을 벡터 검색과 비교해 봅니다
import numpy as np
gvecs = embed(graph_docs)

def gvector_search(query, n=2):
    sims = gvecs @ embed([query])[0]
    return [graph_docs[i] for i in np.argsort(-sims)[:n]]

print("벡터 Top-2:", gvector_search(q))
# 코퍼스가 작으면 벡터도 두 문서를 함께 가져올 수 있습니다.
# 하지만 문서가 수천 개로 늘면 '두 조각이 같이 Top-K에 들' 확률이 낮아지고,
# 그래프는 길(X-200 → 아크메전자 → 김철수)을 따라 확정적으로 연결합니다.

> 🎯 **도전 과제** — 슬라이드 '어떤 주제를 그래프로?'에서 고른 주제(조직도·부품 카탈로그·드라마 대본·뉴스 아카이브…)로
> `graph_docs`를 4~6문장 직접 써서 바꾸고, 멀티홉 질문을 하나 만들어 다시 실행해 보세요.

## 산출물 체크리스트
- ✅ BM25 + 벡터 **하이브리드(RRF)** 동작
- ✅ **Reranker** 2단계(N→K)
- ✅ HyDE·Multi-Query로 모호 질문 보강
- ✅ **질문 8개** 기준선 대비 품질 비교 메모
- ✅ **미니 GraphRAG** — 트리플 추출 → 2-hop 탐색 → 멀티홉 답변 (+그래프감 주제 판별)

> 💡 이 `advanced_rag()`가 Week 3에서 **에이전트의 검색 도구**가 됩니다.